<a href="https://colab.research.google.com/github/BASE-Laboratory/OpenImpala/blob/master/tutorials/06_topology_optimisation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutorial 6: Topology Optimisation (Proof of Concept)

Can we find the voxel arrangement that minimises tortuosity for a given porosity? This tutorial demonstrates a simple hill-climbing optimisation where OpenImpala serves as the cost-function evaluator.

The algorithm randomly swaps solid and pore voxels (keeping the total porosity strictly fixed) and accepts each swap only if it reduces the tortuosity. This is a basic stochastic search — not a state-of-the-art optimisation method — but it illustrates the key idea: OpenImpala can be called repeatedly inside an optimisation loop without stability or performance issues.

**What you will learn:**
1. Initialise a random 50/50 microstructure.
2. Run a stochastic hill-climbing loop with OpenImpala in the inner loop.
3. Plot the optimisation trajectory and inspect the resulting structure.

> **Scope:** We use a small domain ($24^3$) and 80 iterations to keep run times under a few minutes on a laptop. On a real problem you would use a larger domain, more iterations, and a more sophisticated algorithm (e.g. simulated annealing, genetic algorithms, or gradient-based methods). The purpose here is to validate that OpenImpala is stable under repeated calls and to show the integration pattern.

In [ ]:
# Install OpenImpala and visualization libraries
!pip install -q openimpala matplotlib

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import openimpala as oi

print(f"OpenImpala version {oi.__version__} loaded.")

SIZE = 24        # Domain side length (24^3 = 13,824 voxels)
ITERATIONS = 80  # Number of swap attempts

# Initialise a random 50/50 microstructure
np.random.seed(42)
micro = np.random.choice([0, 1], size=(SIZE, SIZE, SIZE)).astype(np.int32)

In [ ]:
best_tau = np.inf
tau_history = []

print("Starting hill-climbing optimisation...")
t_start = time.time()

with oi.Session():
    # Evaluate initial state
    perc = oi.percolation_check(micro, phase=1, direction="z")
    if perc.percolates:
        best_tau = oi.tortuosity(micro, phase=1, direction="z").tortuosity

    print(f"Initial tortuosity: {best_tau:.4f}")
    tau_history.append(best_tau)

    accepted = 0
    for i in range(ITERATIONS):
        # Find all current pore (1) and solid (0) voxels
        pore_coords = np.argwhere(micro == 1)
        solid_coords = np.argwhere(micro == 0)

        # Pick one random pore and one random solid voxel
        p_idx = pore_coords[np.random.choice(len(pore_coords))]
        s_idx = solid_coords[np.random.choice(len(solid_coords))]

        # Swap them (porosity stays exactly the same)
        micro[tuple(p_idx)] = 0
        micro[tuple(s_idx)] = 1

        # Evaluate the new configuration
        perc = oi.percolation_check(micro, phase=1, direction="z")
        new_tau = np.inf
        if perc.percolates:
            new_tau = oi.tortuosity(micro, phase=1, direction="z").tortuosity

        # Accept only if transport improved
        if new_tau < best_tau:
            best_tau = new_tau
            accepted += 1
            if accepted <= 10 or accepted % 5 == 0:
                print(f"  Iteration {i+1:3d} | Accepted swap #{accepted} | tau = {best_tau:.4f}")
        else:
            # Revert the swap
            micro[tuple(p_idx)] = 1
            micro[tuple(s_idx)] = 0

        tau_history.append(best_tau)

t_total = time.time() - t_start
print(f"\nCompleted {ITERATIONS} iterations in {t_total:.2f}s.")
print(f"Accepted {accepted}/{ITERATIONS} swaps.")
print(f"Tortuosity: {tau_history[0]:.4f} -> {best_tau:.4f} "
      f"({(1 - best_tau / tau_history[0]) * 100:.1f}% reduction)")

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(14, 4.5), dpi=120)

# Optimisation trajectory
ax1.plot(tau_history, '-', lw=2, color='#2ca02c')
ax1.set_xlabel("Iteration")
ax1.set_ylabel(r"Tortuosity ($\tau$)")
ax1.set_title("Optimisation Trajectory")
ax1.grid(True, alpha=0.3)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# Initial structure (regenerate from same seed for comparison)
np.random.seed(42)
initial = np.random.choice([0, 1], size=(SIZE, SIZE, SIZE)).astype(np.int32)
ax2.imshow(initial[:, SIZE//2, :], cmap='gray', interpolation='nearest')
ax2.set_title("Initial (random)")
ax2.axis('off')

# Optimised structure
ax3.imshow(micro[:, SIZE//2, :], cmap='gray', interpolation='nearest')
ax3.set_title("After optimisation")
ax3.axis('off')

plt.tight_layout()
plt.show()

## Limitations and Extensions

The hill-climbing algorithm used here is a greedy local search — it will get stuck in local optima and has no mechanism for escaping them. Single-voxel swaps also make very small changes to the geometry, limiting the structural rearrangements that are reachable.

For real applications, consider:
- **Simulated annealing** — accepts some worsening moves early on to escape local optima.
- **Genetic algorithms** — evolves a population of candidate structures.
- **Gradient-based topology optimisation** — uses sensitivity analysis (e.g. adjoint methods) to update the structure more efficiently.
- **Multi-voxel moves** — swapping clusters or applying morphological operations (erosion, dilation) per step.

The key takeaway is that OpenImpala is stable and efficient enough to be called hundreds of times inside such loops, making it a practical building block for computational design workflows.

## Next Steps

- [Tutorial 7: Scaling to HPC](07_hpc_scaling.ipynb) — Run larger optimisation campaigns on a cluster.
- [Tutorial 1: Getting Started](01_hello_openimpala.ipynb) — Review the basics.